## marital status

In [17]:
import pandas as pd
import re
import json

In [18]:
# import dictionary of education(json)

with open(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\marital_status\marital_status_pattern.json",encoding='utf-8')as fin:
    pattern_dict = json.load(fin)

In [19]:
# import data

data_marital = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\marital_status\1_customer_comp_marital_status.csv" , encoding = 'utf8')

data_marital.shape

(48, 7)

In [20]:
pd.set_option('display.max_rows', None)

In [21]:
data_marital.head(5)

,subject,no,id,case,detail case,flag,marital_status
0,marital_status,1,marital_status-ENG-1,ENG,normal,good,Single
1,marital_status,2,marital_status-ENG-2,ENG,normal,good,Married
2,marital_status,3,marital_status-ENG-3,ENG,normal,good,Widowed
3,marital_status,4,marital_status-ENG-4,ENG,normal,good,Children_under_15
4,marital_status,5,marital_status-ENG-5,ENG,normal,good,S


In [22]:
data_marital.tail(5)

,subject,no,id,case,detail case,flag,marital_status
43,marital_status,44,marital_status-THA-44,THA,spell,bad,อายุต่ำกว่า 15 ปี
44,marital_status,45,marital_status-THA-45,THA,spell,bad,โส
45,marital_status,46,marital_status-THA-46,THA,spell,bad,สมร
46,marital_status,47,marital_status-THA-47,THA,spell,bad,หยาราง
47,marital_status,48,marital_status-THA-48,THA,spell,bad,เดกอายตกวา 15 ป


In [23]:
# make the data to lower case
# remove .
# Consecutively characters convert to one-character
def standard_transform(text_input):
    text_input = text_input.lower().replace('.' , '')
    text_input = re.sub(r'([a-zก-๛])\1+' , r'\1' , text_input)
    return text_input



# create column for transform data
data_marital['marital_status_transform'] = data_marital['marital_status'].apply(lambda row:standard_transform(row))

In [24]:
# choose only english letters
def take_eng(input_marital) :
    try :
        if re.search(r'[a-zA-Z]+' , input_marital):
            return input_marital
    except AttributeError :
        return None
    
    
    
# create column for english letters
data_marital['df_eng'] = data_marital['marital_status_transform'].apply(lambda marital_transform : take_eng(marital_transform))

In [25]:
# choose only thai letters
def take_thai(input_marital) :
    try :
        if re.search(r'[ก-๛]+' , input_marital) :
            return input_marital
    except AttributeError :
        return None
    
    
    
# create column for thai letters
data_marital['df_thai'] = data_marital['marital_status_transform'].apply(lambda marital_transform : take_thai(marital_transform))

In [26]:
# create data frame for case = eng and case = thai
status_eng = data_marital[data_marital['df_eng'].notnull()]
status_thai = data_marital[data_marital['df_thai'].notnull()]

# del original data frame
del data_marital

In [27]:
# create standard format
def std_status(text_input,lang) :
    for pattern,value in pattern_dict[lang].items():
        if re.search(pattern , text_input):
            return value
        else:
            continue
    return text_input



# create column for standard format
status_eng['standard_marital_status'] = status_eng['df_eng'].apply(lambda df_eng : std_status(df_eng,'eng'))
status_thai['standard_marital_status'] = status_thai['df_thai'].apply(lambda df_thai : std_status(df_thai , 'tha'))

In [28]:
# combined data frame of english and thai
marital_status = pd.concat([status_eng , status_thai])

In [29]:
# standard output / status
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    
    
    
marital_status[['std_output' , 'status']] = marital_status.apply(lambda row: pd.Series(result(row['marital_status'], row['standard_marital_status'])), axis=1)

In [31]:
selected_cols = ['id','marital_status','standard_marital_status' ,'std_output' ,'status']

marital_status = marital_status[selected_cols]

In [32]:
marital_status.tail(5)

,id,marital_status,standard_marital_status,std_output,status
43,marital_status-THA-44,อายุต่ำกว่า 15 ปี,Children_under_15,Children_under_15,2
44,marital_status-THA-45,โส,Single,Single,2
45,marital_status-THA-46,สมร,Married,Married,2
46,marital_status-THA-47,หยาราง,Widowed,Widowed,2
47,marital_status-THA-48,เดกอายตกวา 15 ป,Children_under_15,Children_under_15,2
